In [3]:
#pip install pandas

In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
from torch.utils.data import DataLoader, TensorDataset
from math import ceil


In [5]:
datapath = "C:\\Users\\Yiran\\OneDrive - UW\\Simulation\\AIdgpgen\\"
obs_files = []
for datafile in os.listdir(datapath):
    if '20170' in datafile and '.csv' in datafile:
        obs_files.append(os.path.join(datapath, datafile))


In [6]:
def dataprocess(filepath, dimension_need=True):
    df_test = pd.read_csv(filepath)
    #time process
    time_decode = lambda x: int(x.split(":")[0])*12+int(x.split(":")[1])//5

    #taz process
    taz_values = sorted(df_test.taz.unique())
    taz_map = {value: index for index, value in enumerate(taz_values)}
    taz_decode = lambda x: taz_map[x]

    #ids process
    newid_values = sorted(df_test.newid.unique())
    newid_map = {value: index for index, value in enumerate(newid_values)}
    newid_decode = lambda x: newid_map[x]

    df_test['newid_idx'] = df_test['newid'].apply(newid_decode)
    df_test['taz_idx'] = df_test['taz'].apply(taz_decode)
    df_test['time_idx'] = df_test['time'].apply(time_decode)

    num_id, num_time, num_taz = len(df_test.newid_idx.unique()), len(df_test.time_idx.unique()), len(df_test.taz_idx.unique())

    df_test = df_test[['newid_idx', 'taz_idx', 'time_idx', 'sum']]

    if dimension_need:
        return df_test, num_id, num_time, num_taz
    return df_test

In [7]:
df_test, num_id, num_time, num_taz = dataprocess(obs_files[0])

In [8]:
tensor = np.full((num_id, num_time, num_taz), np.nan)

for row in df_test.itertuples():
    tensor[row.newid_idx, row.time_idx, row.taz_idx] = 1 #or row.sum


In [9]:
# Flatten the matrix to (num_id, 288 * 167)
flattened_tensor = tensor.reshape(tensor.shape[0], -1)

# Create mask for observed values (1 = observed, 0 = missing)
mask = ~np.isnan(flattened_tensor)

# Replace NaNs with 0 for now (you'll use the mask during training)
flattened_tensor[np.isnan(flattened_tensor)] = 0

# Convert to PyTorch tensors
X_tensor = torch.tensor(flattened_tensor, dtype=torch.float32)
mask_tensor = torch.tensor(mask, dtype=torch.bool)


In [15]:
from torch.utils.data import TensorDataset, DataLoader
batch_size = 32
dataset = TensorDataset(X_tensor, mask_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [11]:
class Generator(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, output_dim),
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.net(z)
    
class Critic(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        return self.net(x)


In [12]:
def compute_gradient_penalty(critic, real, fake, device):
    alpha = torch.rand(real.size(0), 1).to(device)
    alpha = alpha.expand_as(real)

    interpolates = (alpha * real + ((1 - alpha) * fake)).requires_grad_(True)
    d_interpolates = critic(interpolates)
    ones = torch.ones(d_interpolates.size()).to(device)

    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=ones,
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty


In [16]:
latent_dim = 100
output_dim = num_time * num_taz

device = torch.device("gpu")  # or "cuda" if you later switch to GPU

G = Generator(latent_dim, output_dim).to(device)
C = Critic(output_dim).to(device)

g_optimizer = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.0, 0.9))
c_optimizer = torch.optim.Adam(C.parameters(), lr=1e-4, betas=(0.0, 0.9))

epochs = 100
critic_steps = 5
lambda_gp = 10

for epoch in range(epochs):
    for i, (real_batch, mask_batch) in enumerate(dataloader):
        real_batch = real_batch.to(device)
        mask_batch = mask_batch.to(device)

        for _ in range(critic_steps):
            z = torch.randn(real_batch.size(0), latent_dim).to(device)
            fake = G(z).detach()

            c_optimizer.zero_grad()
            loss_real = -C(real_batch).mean()
            loss_fake = C(fake).mean()
            gp = compute_gradient_penalty(C, real_batch, fake, device)
            critic_loss = loss_real + loss_fake + lambda_gp * gp
            critic_loss.backward()
            c_optimizer.step()

        # Generator update
        z = torch.randn(batch_size, latent_dim).to(device)
        fake = G(z)
        g_loss = -C(fake).mean()

        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

    print(f"[Epoch {epoch+1}/{epochs}] D_loss: {critic_loss.item():.2f}, G_loss: {g_loss.item():.2f}")


[Epoch 1/100] D_loss: -0.29, G_loss: 0.15
[Epoch 2/100] D_loss: -0.30, G_loss: 0.76
[Epoch 3/100] D_loss: -0.81, G_loss: 1.50
[Epoch 4/100] D_loss: -0.75, G_loss: 1.50
[Epoch 5/100] D_loss: -0.41, G_loss: 1.77
[Epoch 6/100] D_loss: -0.85, G_loss: 2.16
[Epoch 7/100] D_loss: -0.88, G_loss: 1.72
[Epoch 8/100] D_loss: -0.98, G_loss: 1.48
[Epoch 9/100] D_loss: -0.82, G_loss: 1.96
[Epoch 10/100] D_loss: -0.91, G_loss: 2.07
[Epoch 11/100] D_loss: -0.80, G_loss: 1.58
[Epoch 12/100] D_loss: -1.41, G_loss: 1.36
[Epoch 13/100] D_loss: -0.73, G_loss: 1.23
[Epoch 14/100] D_loss: -0.44, G_loss: 0.97
[Epoch 15/100] D_loss: -1.68, G_loss: 2.07
[Epoch 16/100] D_loss: -1.28, G_loss: 1.28
[Epoch 17/100] D_loss: -1.27, G_loss: 1.59
[Epoch 18/100] D_loss: -1.50, G_loss: 0.01
[Epoch 19/100] D_loss: -3.56, G_loss: 1.88
[Epoch 20/100] D_loss: -1.60, G_loss: 2.16
[Epoch 21/100] D_loss: -1.57, G_loss: 3.55
[Epoch 22/100] D_loss: -1.72, G_loss: 3.02
[Epoch 23/100] D_loss: -1.48, G_loss: 2.96
[Epoch 24/100] D_los

In [35]:
torch.save(G.state_dict(), 'generator_v1.pt')
torch.save(C.state_dict(), 'critic_v1.pt')

In [39]:
# Save config as a JSON or simple .txt file
generator_config = {
    "latent_dim": 100,
    "output_dim": 288 * 167,
    "activation": "ReLU",
    "hidden_layers": [512, 1024]
}

import json
with open(datapath+"generator_config_v1.json", "w") as f:
    json.dump(generator_config, f)


In [19]:
G.eval()
with torch.no_grad():
    z = torch.randn(10000, latent_dim).to(device)
    generated = G(z).view(-1, num_time, num_taz).cpu().numpy()
    generated_binary = (generated > 0.5).astype(int)  # optional

In [22]:
generated_binary.shape

(10000, 288, 167)

In [25]:

# --------------------------
# Device Setup
# --------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------
# Dimensions
# --------------------------
latent_dim = 30
num_time = 288
num_taz = 167
output_dim = num_time * num_taz
batch_size = 64
critic_steps = 3
lambda_gp = 10
learning_rate = 3e-4
total_gen_steps = 12000  # total generator iterations

# --------------------------
# Generator
# --------------------------
class Generator(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 100),
            nn.ReLU(),
            nn.Linear(100, 300),
            nn.ReLU(),
            nn.Linear(300, output_dim),
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.model(z)

# --------------------------
# Critic
# --------------------------
class Critic(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 300),
            nn.LeakyReLU(0.2),
            nn.Linear(300, 100),
            nn.LeakyReLU(0.2),
            nn.Linear(100, 50),
            nn.LeakyReLU(0.2),
            nn.Linear(50, 1)
        )

    def forward(self, x):
        return self.model(x)

# --------------------------
# Gradient Penalty
# --------------------------
def compute_gradient_penalty(critic, real, fake, device):
    alpha = torch.rand(real.size(0), 1).to(device)
    alpha = alpha.expand_as(real)
    interpolates = (alpha * real + ((1 - alpha) * fake)).requires_grad_(True)
    d_interpolates = critic(interpolates)
    ones = torch.ones(d_interpolates.size()).to(device)

    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=ones,
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

# --------------------------
# DataLoader
# --------------------------
# Assume `tensor` is (13709, 288, 167) with np.nan for missing
#tensor = np.full((13709, 288, 167), np.nan)
#for row in df_test.itertuples():
#    tensor[row.newid_idx, row.time_idx, row.taz_idx] = 1
dataset = TensorDataset(X_tensor, mask_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [27]:
# Derive epochs
steps_per_epoch = ceil(len(dataloader))  # ~13709 / 64 = 215
epochs = ceil(total_gen_steps / steps_per_epoch)

# --------------------------
# Training
# --------------------------
G_v2 = Generator(latent_dim, output_dim).to(device)
C_v2 = Critic(output_dim).to(device)
g_optimizer = torch.optim.Adam(G_v2.parameters(), lr=learning_rate, betas=(0.0, 0.9))
c_optimizer = torch.optim.Adam(C_v2.parameters(), lr=learning_rate, betas=(0.0, 0.9))

for epoch in range(epochs):
    for real_batch, mask_batch in dataloader:
        real_batch = real_batch.to(device)
        mask_batch = mask_batch.to(device)

        for _ in range(critic_steps):
            z = torch.randn(real_batch.size(0), latent_dim).to(device)
            fake = G_v2(z).detach()

            c_optimizer.zero_grad()
            loss_real = -C_v2(real_batch).mean()
            loss_fake = C_v2(fake).mean()
            gp = compute_gradient_penalty(C_v2, real_batch, fake, device)
            critic_loss = loss_real + loss_fake + lambda_gp * gp
            critic_loss.backward()
            c_optimizer.step()

        # Generator update
        z = torch.randn(batch_size, latent_dim).to(device)
        fake = G_v2(z)
        g_loss = -C_v2(fake).mean()

        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

    print(f"[Epoch {epoch+1}/{epochs}] D_loss: {critic_loss.item():.4f}, G_loss: {g_loss.item():.4f}")


C:\Users\Yiran\anaconda3\envs\torch-gpu\lib\site-packages\torch\autograd\graph.py:824: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:181.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[Epoch 1/56] D_loss: 0.0134, G_loss: 0.1328
[Epoch 2/56] D_loss: 0.0059, G_loss: 0.2579
[Epoch 3/56] D_loss: -0.0810, G_loss: 0.4455
[Epoch 4/56] D_loss: -0.1060, G_loss: 0.6846
[Epoch 5/56] D_loss: -0.2701, G_loss: 0.8980
[Epoch 6/56] D_loss: -0.2216, G_loss: 0.9838
[Epoch 7/56] D_loss: -0.0691, G_loss: 1.1093
[Epoch 8/56] D_loss: -0.1763, G_loss: 1.1683
[Epoch 9/56] D_loss: -0.7706, G_loss: 1.8168
[Epoch 10/56] D_loss: -0.6788, G_loss: 1.2025
[Epoch 11/56] D_loss: -0.4287, G_loss: 1.5063
[Epoch 12/56] D_loss: -0.4153, G_loss: 1.6971
[Epoch 13/56] D_loss: -0.6041, G_loss: 2.2216
[Epoch 14/56] D_loss: -0.0212, G_loss: 1.1325
[Epoch 15/56] D_loss: -0.6008, G_loss: 1.5421
[Epoch 16/56] D_loss: -0.5793, G_loss: 1.4462
[Epoch 17/56] D_loss: -0.4595, G_loss: 1.3884
[Epoch 18/56] D_loss: -1.2176, G_loss: 3.3503
[Epoch 19/56] D_loss: -1.1032, G_loss: 2.8942
[Epoch 20/56] D_loss: -0.6593, G_loss: 2.4258
[Epoch 21/56] D_loss: -0.1674, G_loss: 1.7338
[Epoch 22/56] D_loss: -1.1090, G_loss: 2.6440

In [28]:
G_v2.eval()
with torch.no_grad():
    z = torch.randn(10000, latent_dim).to(device)
    generated = G_v2(z).view(-1, num_time, num_taz).cpu().numpy()
    generated_binary_v2 = (generated > 0.5).astype(int)  # optional

In [37]:
torch.save(G_v2.state_dict(), datapath+'generator_v2.pt')
torch.save(C_v2.state_dict(), datapath+'critic_v2.pt')

In [40]:
# Save config as a JSON or simple .txt file
generator_config = {
    "latent_dim": latent_dim,
    "output_dim": 288 * 167,
    "activation": "ReLU",
    "hidden_layers": [100, 300]
}

import json
with open(datapath+"generator_config_v2.json", "w") as f:
    json.dump(generator_config, f)


In [17]:
with open(datapath+'running stauts.txt', 'w') as f:
    f.write('running test, done\n')

f.close()

In [34]:
print(np.sum(generated_binary), np.sum(generated_binary_v2))

195835 370247
